# DementiaBank + DementiaNet (HC vs MCI)

This notebook combines audio from `dementiabank` and `dementianet`, maps labels to `HC`/`MCI`, extracts segment-level eGeMAPS features, and trains a model like `test.ipynb`.

In [1]:
from pathlib import Path
import tempfile
import re

import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

segments_df = None

DEMENTIANET_BASE = Path("/Users/tarinpairor/Downloads/dementianet")

def map_label(raw_label: str) -> str:
    raw = str(raw_label).strip().lower()
    if raw in {"mci", "dementia", "ad"}:
        return "MCI"
    if raw in {"hc", "control", "nodementia", "no_dementia"}:
        return "HC"
    if "dementia" in raw and "no" not in raw:
        return "MCI"
    return "HC"


def collect_audio_index():
    rows = []
    dementia_dir = DEMENTIANET_BASE / "dementia"
    nodementia_dir = DEMENTIANET_BASE / "nodementia"

    for ext in ("*.wav", "*.m4a", "*.mp3"):
        for p in sorted(dementia_dir.rglob(ext)):
            rows.append(
                {
                    "dataset": "dementianet",
                    "raw_label": "Dementia",
                    "label": "MCI",
                    "path": p,
                }
            )
        for p in sorted(nodementia_dir.rglob(ext)):
            rows.append(
                {
                    "dataset": "dementianet",
                    "raw_label": "Control",
                    "label": "HC",
                    "path": p,
                }
            )

    return rows


AUDIO_INDEX = collect_audio_index()
print(f"Total indexed audio files: {len(AUDIO_INDEX)}")
print(f"From dementianet: {sum(1 for x in AUDIO_INDEX if x['dataset'] == 'dementianet')}")
missing = [x for x in AUDIO_INDEX if not x['path'].exists()]
print(f"Missing files: {len(missing)}")
for item in missing:
    print(f"MISSING -> {item['dataset']} | {item['raw_label']} | {item['path']}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.11/site-p

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.11/site-p

AttributeError: _ARRAY_API not found

Total indexed audio files: 149
From dementianet: 149
Missing files: 0


In [ ]:
import matplotlib.pyplot as plt


def summarize_audio(path: Path):
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    dur = len(y) / sr
    peak = float(np.max(np.abs(y)))
    rms = float(np.sqrt(np.mean(y ** 2)))
    print(f"{path.name}: duration={dur:.2f}s, sr={sr} Hz, peak={peak:.4f}, RMS={rms:.4f}")

    max_pts = 50_000
    if len(y) > max_pts:
        step = len(y) // max_pts
        y_plot = y[::step]
        t = np.arange(len(y_plot)) * step / sr
    else:
        y_plot = y
        t = np.arange(len(y)) / sr
    fig, ax = plt.subplots(figsize=(10, 2))
    ax.plot(t, y_plot, lw=0.3)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.set_title(f"Waveform - {path.name}")
    plt.tight_layout()
    plt.show()
    return {"y": y, "sr": sr, "path": path}


loaded = {}
failed_files = []
print("Per-file analysis (summary + waveform):")
for i, item in enumerate(AUDIO_INDEX, start=1):
    source_key = f"{item['dataset']}::{item['path'].stem}"
    print(f"[{i}/{len(AUDIO_INDEX)}] --- {source_key} ({item['label']}) ---")
    try:
        info = summarize_audio(item["path"])
    except Exception as e:
        failed_files.append({"source_key": source_key, "path": str(item["path"]), "error": str(e)})
        print(f"SKIP (read/plot error): {e}")
        print()
        continue

    if info is not None:
        info["label"] = item["label"]
        info["raw_label"] = item["raw_label"]
        info["source_dataset"] = item["dataset"]
        loaded[source_key] = info
    print()

print(f"Loaded files: {len(loaded)}")
print(f"Failed files: {len(failed_files)}")
if failed_files:
    display(pd.DataFrame(failed_files).head(20))


In [4]:
import pandas as pd

# Convert AUDIO_INDEX (list of dicts) to a DataFrame for grouping/counting
audio_index_df = pd.DataFrame(AUDIO_INDEX)
label_counts = audio_index_df.groupby("label").size()
print(label_counts)

label
HC     89
MCI    60
dtype: int64


In [5]:
import numpy as np

MIN_SILENCE_SEC = 0.5
FRAME_MS = 30.0
HOP_MS = 10.0
SILENCE_DB_BELOW_PEAK = 20.0


def frame_energy_db(y, sr):
    fl = max(1, int(sr * FRAME_MS / 1000.0))
    hop = max(1, int(sr * HOP_MS / 1000.0))
    if len(y) < fl:
        return np.array([0.0]), fl, hop
    peak = float(np.max(np.abs(y))) + 1e-12
    n_frames = 1 + (len(y) - fl) // hop
    levels = []
    for i in range(n_frames):
        start = i * hop
        frame = y[start : start + fl]
        rms = float(np.sqrt(np.mean(frame ** 2)))
        db = 20.0 * np.log10(rms / peak + 1e-12)
        levels.append(db)
    return np.array(levels), fl, hop


def split_on_silence(y, sr, min_silence_sec=MIN_SILENCE_SEC, silence_db_below_peak=SILENCE_DB_BELOW_PEAK):
    db_frames, fl, hop = frame_energy_db(y, sr)
    silent = db_frames < -silence_db_below_peak

    regions = []
    i = 0
    n = len(silent)
    while i < n:
        if silent[i]:
            i += 1
            continue
        j = i
        while j < n and not silent[j]:
            j += 1
        start_s = i * hop
        end_s = min(len(y), j * hop + fl)
        regions.append((start_s, end_s))
        i = j

    if not regions:
        return [(0, len(y))]

    merged = [regions[0]]
    for (s, e) in regions[1:]:
        ps, pe = merged[-1]
        if (s - pe) / sr < min_silence_sec:
            merged[-1] = (ps, e)
        else:
            merged.append((s, e))
    return merged


In [6]:
import opensmile

FEATURE_COLUMNS = [
    "F0semitoneFrom27.5Hz_sma3nz_amean",
    "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
    "loudness_sma3_amean",
    "loudness_sma3_stddevNorm",
    "HNRdBACF_sma3nz_amean",
    "mfcc1_sma3_amean",
    "mfcc2_sma3_amean",
]


def build_smile() -> opensmile.Smile:
    return opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.Functionals,
    )


def extract_egemaps_functionals(audio_path, smile=None):
    smile = smile or build_smile()
    feats = smile.process_file(str(audio_path)).reset_index(drop=True)
    return feats


def extract_feature_columns(audio_path, smile=None):
    smile = smile or build_smile()
    feats = extract_egemaps_functionals(audio_path, smile=smile)
    missing = [c for c in FEATURE_COLUMNS if c not in feats.columns]
    if missing:
        raise ValueError(f"Missing columns in openSMILE output: {missing}")
    return feats[FEATURE_COLUMNS].copy()


def build_segment_table(loaded, smile=None):
    smile = smile or build_smile()
    rows = []
    tmpdir = tempfile.mkdtemp(prefix="dementiabank_seg_v2_")
    tmp_base = Path(tmpdir)

    for source_key, info in loaded.items():
        safe_key = re.sub(r"[^A-Za-z0-9._-]+", "_", str(source_key))
        y = np.asarray(info["y"], dtype=np.float32)
        sr = int(info["sr"])
        path = info["path"]
        label = info["label"]
        source_dataset = info["source_dataset"]
        intervals = split_on_silence(y, sr)

        for seg_i, (start, end) in enumerate(intervals):
            seg = y[start:end]
            if seg.size < sr * 0.05:
                continue
            out_path = tmp_base / f"{safe_key}_seg{seg_i:03d}.wav"
            sf.write(out_path, seg, sr)
            feat_row = extract_feature_columns(out_path, smile=smile).iloc[0].to_dict()
            row = {
                "source_key": source_key,
                "source_dataset": source_dataset,
                "segment_index": seg_i,
                "label": label,
                "duration_sec": len(seg) / sr,
                "segment_wav_path": str(out_path),
                "original_audio": str(path),
            }
            row.update(feat_row)
            rows.append(row)

    return pd.DataFrame(rows), tmp_base


if loaded:
    segments_df, _tmp = build_segment_table(loaded)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    print(f"Total segments: {len(segments_df)}")
    display(segments_df)
else:
    segments_df = pd.DataFrame()
    print("No audio loaded; check file paths.")


KeyboardInterrupt: 

In [17]:
# Export segments_df to a CSV file
segments_df.to_csv("segments_df.csv", index=False)

# Load directly from here

In [14]:
# Load directly from here

segments_df = pd.read_csv('segments_df.csv')
segments_df

,source_key,source_dataset,segment_index,label,duration_sec,segment_wav_path,original_audio,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,loudness_sma3_amean,loudness_sma3_stddevNorm,HNRdBACF_sma3nz_amean,mfcc1_sma3_amean,mfcc2_sma3_amean
0,dementianet::AbeBurrows_5,dementianet,0,MCI,5.51,/var/folders/h1/_n_bj8qj1nd9nhqqdqkdgdt40000gn...,/Users/tarinpairor/Downloads/dementianet/demen...,27.012375,0.243077,0.893451,0.606037,3.649729,24.786909,20.573875
1,dementianet::AbeBurrows_5,dementianet,1,MCI,2.65,/var/folders/h1/_n_bj8qj1nd9nhqqdqkdgdt40000gn...,/Users/tarinpairor/Downloads/dementianet/demen...,28.615625,0.203176,1.072411,0.541246,3.054514,20.093721,19.037392
2,dementianet::AbeBurrows_5,dementianet,2,MCI,0.62,/var/folders/h1/_n_bj8qj1nd9nhqqdqkdgdt40000gn...,/Users/tarinpairor/Downloads/dementianet/demen...,25.071545,0.244726,1.081569,0.600989,0.751346,20.169146,19.687019
3,dementianet::AbeBurrows_5,dementianet,3,MCI,4.90,/var/folders/h1/_n_bj8qj1nd9nhqqdqkdgdt40000gn...,/Users/tarinpairor/Downloads/dementianet/demen...,27.614748,0.144642,0.842000,0.627142,4.216394,22.985872,19.730190
4,dementianet::AbeBurrows_5,dementianet,4,MCI,2.09,/var/folders/h1/_n_bj8qj1nd9nhqqdqkdgdt40000gn...,/Users/tarinpairor/Downloads/dementianet/demen...,26.957123,0.120093,0.975068,0.490646,4.847751,27.481073,20.856985
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2681,dementianet::DominicChianese_3,dementianet,16,HC,0.30,/var/folders/h1/_n_bj8qj1nd9nhqqdqkdgdt40000gn...,/Users/tarinpairor/Downloads/dementianet/nodem...,44.017670,0.369415,0.274213,0.443850,2.808491,10.484063,16.582077
2682,dementianet::DominicChianese_3,dementianet,17,HC,0.05,/var/folders/h1/_n_bj8qj1nd9nhqqdqkdgdt40000gn...,/Users/tarinpairor/Downloads/dementianet/nodem...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2683,dementianet::DominicChianese_3,dementianet,18,HC,0.10,/var/folders/h1/_n_bj8qj1nd9nhqqdqkdgdt40000gn...,/Users/tarinpairor/Downloads/dementianet/nodem...,25.363110,0.005534,0.356812,0.147700,6.779287,43.049515,9.999340
2684,dementianet::DominicChianese_3,dementianet,19,HC,0.27,/var/folders/h1/_n_bj8qj1nd9nhqqdqkdgdt40000gn...,/Users/tarinpairor/Downloads/dementianet/nodem...,23.452593,0.027720,0.239213,0.561806,2.580958,31.632740,4.492559


In [15]:
segments_df.groupby('label').size()

label
HC     1622
MCI    1064
dtype: int64

In [ ]:
# Play four segments with the smallest duration_sec (across all sources)
print("Four segments with the smallest duration_sec:\n")
_df = globals().get("segments_df")
if _df is not None and len(_df) > 0:
    shown = 0
    segs = _df.nsmallest(4, "duration_sec")
    for _, row in segs.iterrows():
        p = Path(row["segment_wav_path"])
        if not p.exists():
            continue
        data, sr = sf.read(p, dtype="float32", always_2d=False)
        if data.ndim > 1:
            data = np.mean(data, axis=1)
        print(
            f"{row['source_key']} ({row['source_dataset']}) -> {row['label']} | "
            f"seg {int(row['segment_index'])} | {row['duration_sec']:.2f}s"
        )
        display(Audio(data, rate=sr))
        shown += 1
    if shown == 0:
        print("No segment WAVs to play.")
else:
    print("Run the previous cell to build segments_df first.")


In [17]:
old_df = segments_df.copy()

In [18]:
segments_df = segments_df[segments_df['duration_sec'] > 0.5]

In [19]:
# Reduce the number of "MCI" labels to a factor (e.g., 0.5) of the total "HC" count
# mci_factor = 0.5
# hc_count = segments_df[segments_df['label'] == 'HC'].shape[0]
# mci_count = int(hc_count * mci_factor)
# mci_df = segments_df[segments_df['label'] == 'MCI'].sample(n=mci_count, random_state=42)
# hc_df = segments_df[segments_df['label'] == 'HC']
# segments_df_reduced = pd.concat([hc_df, mci_df], axis=0).reset_index(drop=True)
# segments_df = segments_df_reduced
# segments_df

In [20]:
segments_df.groupby('label').size()

label
HC     1179
MCI     730
dtype: int64

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler

if segments_df is None or len(segments_df) == 0:
    raise ValueError("segments_df is empty. Run the feature extraction cell first.")

model_df = segments_df.copy()
model_df = model_df[model_df["label"].isin(["HC", "MCI"])]

X = model_df[FEATURE_COLUMNS]
y = model_df["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = RandomForestClassifier()
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
y_pred = cross_val_predict(model, X_scaled, y, cv=cv)

print(classification_report(y, y_pred))


              precision    recall  f1-score   support

          HC       0.73      0.90      0.81      1179
         MCI       0.74      0.47      0.58       730

    accuracy                           0.73      1909
   macro avg       0.74      0.68      0.69      1909
weighted avg       0.74      0.73      0.72      1909



In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import numpy as np

if segments_df is None or len(segments_df) == 0:
    raise ValueError("segments_df is empty. Run the feature extraction cell first.")

model_df = segments_df.copy()
model_df = model_df[model_df["label"].isin(["HC", "MCI"])]

X = model_df[FEATURE_COLUMNS]
y = model_df["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = RandomForestClassifier()
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')

print(f"Cross-validation accuracies for each fold:\n{scores}")
print(f"\nMean accuracy: {np.mean(scores):.4f} ± {np.std(scores):.4f}")


Cross-validation accuracies for each fold:
[0.72251309 0.7434555  0.7434555  0.7434555  0.71204188 0.72251309
 0.7539267  0.71727749 0.70680628 0.72105263]

Mean accuracy: 0.7286 ± 0.0152


# Record and run model

In [ ]:
def record_and_predict_hc_mci(feature_columns=FEATURE_COLUMNS, model=None, scaler=None, duration=5, temp_filename="live_record_audio.wav"):
    """
    Record audio via the microphone for `duration` seconds, then predict whether the audio is HC or MCI.
    """
    import sounddevice as sd
    import scipy.io.wavfile as wav
    import numpy as np
    import opensmile
    import io
    import IPython.display as ipd

    # 1. Record live audio
    fs = 16000  # Sample rate
    print(f"Recording for {duration} seconds. Please speak now...")
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='int16')
    sd.wait()
    wav.write(temp_filename, fs, audio)
    print("Recording finished.")

    # Play back the audio
    display(ipd.Audio(temp_filename, rate=fs))

    # 2. Feature extraction using opensmile
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.Functionals,
    )
    features = smile.process_file(temp_filename)
    features = features.reset_index(drop=True)
    
    # Ensure columns match the training FEATURE_COLUMNS
    for col in feature_columns:
        if col not in features.columns:
            features[col] = np.nan  # Fill missing columns as NaN
    features = features[feature_columns]

    # 3. Fit or use existing scaler/model
    if scaler is None or model is None:
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.preprocessing import StandardScaler
        # Re-fit using all available data
        scaler = StandardScaler()
        X_all = segments_df[feature_columns]
        scaler.fit(X_all)
        model = RandomForestClassifier()
        model.fit(scaler.transform(X_all), segments_df["label"])
    
    # 4. Predict
    X_feat = features.values
    X_scaled = scaler.transform(X_feat)
    pred_label = model.predict(X_scaled)[0]
    proba = model.predict_proba(X_scaled)[0]
    return {
        "predicted_label": pred_label,
        "prob_HC": float(proba[list(model.classes_).index("HC")]) if "HC" in model.classes_ else None,
        "prob_MCI": float(proba[list(model.classes_).index("MCI")]) if "MCI" in model.classes_ else None,
    }

# Example usage in notebook:
result = record_and_predict_hc_mci()
print("Prediction:", result["predicted_label"], "Prob HC:", result["prob_HC"], "Prob MCI:", result["prob_MCI"])

In [ ]:
import joblib

# Save the trained model
joblib.dump(model, "dementiabank_dementianet_model.joblib")
# Save the scaler
joblib.dump(scaler, "dementiabank_dementianet_scaler.joblib")

['dementiabank_dementianet_scaler.joblib']